# 01 — Entraînement du PINN

Ce notebook importe les modules Python du projet et lance un essai d'entraînement.

**Attention :** les paramètres physiques et les conditions aux limites doivent être vérifiés dans le cas OpenFOAM avant toute étude quantitative.

In [1]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
print('Répertoire du projet :', ROOT)

from config import Config
from train import train
from model import SloshingPINN
from visualization import plot_history, animate_comparison

Répertoire du projet : /home/ludodo/SINN


In [2]:
cfg = Config()

# Petit essai fonctionnel. Augmenter ces valeurs ensuite.
cfg.epochs = 100
cfg.n_collocation = 256
cfg.n_initial = 128
cfg.n_wall = 128
cfg.hidden_width = 48
cfg.hidden_layers = 3
cfg.print_every = 1
cfg.use_fourier_features = False
cfg.use_temporal_segmentation = False
cfg.use_adaptive_loss_weights = False
cfg.use_adaptive_sampling = False
cfg.sampling_update_interval = 10
model, history, run_dir = train(cfg, ROOT / "models", model_name="test")

Training config:
	epochs=100
	fourier features=False
	adaptive sampling=False
	adaptive loss weights=False
	use lbfgs=False
	temporal segmentation=False
	windows=1
	epochs_per_window=100
	learning_rate=1.00e-03
	device=cuda
	seed=1234


Training for t=[0.0, 10.0]: 100%|██████████| 100/100 [00:11<00:00,  8.97it/s, continuity=2.2494e-03, lr=1.00e-03, momentum=4.0431e-02, total=9.2064e-01, volume_fraction=1.1944e-03, window=w0]

Training finished. Results saved to: /home/ludodo/SINN/models/test


## Vérifications avant l'entraînement complet

- Vérifier la géométrie et les patchs du domaine.
- Vérifier l'amplitude, la fréquence et la phase du mouvement.
- Remplacer les conditions aux limites génériques par celles du cas réel.
- Ajouter les conditions initiales et aux limites de k et epsilon.
- Tester le signe de la tension superficielle sur un cas simple.
- Étudier la mise à l'échelle et l'équilibrage des pertes.

In [5]:
# Entraînement plus conséquent que le petit essai précédent.
cfg = Config()

# Configuration renforcée : plus de points de collocation, réseau plus profond,
# plus d'épochs pour un entraînement de référence.
cfg.epochs = 20000
cfg.n_collocation = 2048
cfg.n_initial = 512
cfg.n_wall = 512
cfg.hidden_width = 96
cfg.hidden_layers = 5
cfg.print_every = 50

model2, history2, run_dir2 = train(cfg, ROOT / "models")


[     1/20000] total=2.8607e+01 cont=1.488e-02 vof=5.056e-03 mom=2.619e+01 init=4.288e-01 wall=1.289e-01
[    50/20000] total=9.8337e-01 cont=2.079e-04 vof=1.950e-03 mom=2.406e-02 init=1.906e-01 wall=1.969e-03
[   100/20000] total=4.8473e-01 cont=3.416e-04 vof=7.619e-03 mom=8.102e-02 init=7.787e-02 wall=7.851e-04
[   150/20000] total=2.3383e-01 cont=4.195e-04 vof=9.668e-03 mom=5.110e-02 init=3.280e-02 wall=6.373e-04
[   200/20000] total=3.2189e-01 cont=8.257e-04 vof=2.390e-02 mom=1.702e-01 init=2.292e-02 wall=1.515e-03
[   250/20000] total=2.2024e-01 cont=1.374e-03 vof=2.226e-02 mom=6.830e-02 init=2.296e-02 wall=1.049e-03
[   300/20000] total=1.2951e-01 cont=1.945e-03 vof=1.966e-02 mom=1.745e-02 init=1.571e-02 wall=1.590e-03
[   350/20000] total=1.3476e-01 cont=3.817e-03 vof=2.708e-02 mom=4.228e-02 init=1.030e-02 wall=1.351e-03
[   400/20000] total=9.7887e-02 cont=3.841e-03 vof=2.040e-02 mom=1.237e-02 init=1.079e-02 wall=1.626e-03
[   450/20000] total=6.8645e-02 cont=3.568e-03 vof=2.53

In [8]:
# Continuer l'entraînement en chargeant le modèle à 500 epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Nouvelle configuration avec plus d'épochs
cfg_add = Config()
cfg_add.epochs = 2000
cfg_add.n_collocation = 2048
cfg_add.n_initial = 512
cfg_add.n_wall = 512
cfg_add.hidden_width = 96
cfg_add.hidden_layers = 5
cfg_add.print_every = 50

# Charger le modèle entraîné sur 500 epochs via state_path (nouvelle interface)
# checkpoint_path = run_dir2 / "checkpoint_final.pt"
checkpoint_path = ROOT / "models/sloshing_20261028_161714/checkpoint_final.pt"
model_add = SloshingPINN(config=cfg_add, state_path=checkpoint_path).to(device)
# Entraînement supplémentaire
model_add, history_add, run_dir_add = train(model=model_add)


[     1/1000] total=1.0581e-03 cont=7.555e-05 vof=5.837e-04 mom=1.575e-04 init=1.468e-05 wall=8.397e-05
[    50/1000] total=9.4264e-03 cont=1.071e-03 vof=3.068e-03 mom=1.077e-03 init=2.360e-04 wall=1.512e-03
[   100/1000] total=4.4157e-03 cont=3.741e-04 vof=2.425e-03 mom=2.610e-04 init=3.060e-05 wall=6.012e-04
[   150/1000] total=2.5177e-03 cont=1.899e-04 vof=1.362e-03 mom=1.605e-04 init=1.234e-05 wall=3.714e-04
[   200/1000] total=1.8248e-03 cont=1.606e-04 vof=7.745e-04 mom=2.775e-04 init=1.225e-05 wall=2.755e-04
[   250/1000] total=1.1047e-03 cont=1.184e-04 vof=4.805e-04 mom=1.682e-04 init=1.174e-05 wall=1.395e-04
[   300/1000] total=8.9329e-04 cont=7.253e-05 vof=4.168e-04 mom=1.461e-04 init=1.280e-05 wall=9.695e-05
[   350/1000] total=9.6691e-04 cont=8.065e-05 vof=5.357e-04 mom=1.483e-04 init=9.309e-06 wall=7.789e-05
[   400/1000] total=8.2743e-04 cont=8.594e-05 vof=4.114e-04 mom=1.436e-04 init=9.658e-06 wall=6.911e-05
[   450/1000] total=1.0378e-03 cont=1.054e-04 vof=5.898e-04 mom=

In [5]:
of_path = str(ROOT / "data" / "sloshing_data.npz")

animation, fig = animate_comparison(
    of_path,
    model=model2,
)


In [ ]:
plot_history(run_dir2 / "history.json", use_same_figure=True)